In [13]:
# 只需執行一次：把原始 CSV 轉成 Parquet（已完成，勿重跑）
# Parquet 是 columnar 格式，之後的 query 讀取速度比 .csv.gz 快很多

import duckdb

duckdb.sql("""
COPY (SELECT * FROM read_csv_auto('../data/raw/2019-Oct.csv.gz')) -- read_csv_auto：讀取 CSV，自動判斷欄位名稱和資料型態，可直接讀 .gz 壓縮檔
TO '../data/processed/events_2019_10.parquet' (FORMAT PARQUET) -- COPY (...) TO：把 query 結果寫成檔案
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
path = "'../data/processed/events_2019_10.parquet'"
# 把檔案路徑存進變數 path。外層雙引號是 Python 字串，
# 內層單引號會一起放進 SQL，因為 SQL 裡的檔名要用單引號包住

### data profiling

In [14]:
# 1. 欄位名稱與資料型態
duckdb.sql(f"DESCRIBE SELECT * FROM {path}").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_time    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ event_type    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

In [15]:
# 2. 前 10 筆資料，看實際長相
duckdb.sql(f"SELECT * FROM {path} LIMIT 10").show()

┌─────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────────────────────┬──────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │            category_code            │  brand   │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │   int64    │        int64        │               varchar               │ varchar  │ double  │   int64   │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────────────────────┼──────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-10-01 00:00:00 │ view       │   44600062 │ 2103807459595387724 │ NULL                                │ shiseido │   35.79 │ 541312140 │ 72d76fde-8bb3-4e00-8c23-a032dfed738c │
│ 2019-10-01 00:00:00 │ view       │    3900821 │ 2053013552326770905 │ appliances.environ

In [16]:
# 3. 每個欄位的統計摘要：min、max、不重複值數量、平均、missing 比例等
duckdb.sql(f"SUMMARIZE SELECT * FROM {path}").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────────┬──────────────────────────────────────┬──────────────────────────────────────┬───────────────┬────────────────────────────┬───────────────────────┬────────────────────────────┬───────────────────────────┬────────────────────────────┬──────────┬─────────────────┐
│  column_name  │ column_type │                 min                  │                 max                  │ approx_unique │            avg             │          std          │            q25             │            q50            │            q75             │  count   │ null_percentage │
│    varchar    │   varchar   │               varchar                │               varchar                │     int64     │          varchar           │        varchar        │          varchar           │          varchar          │          varchar           │  int64   │  decimal(9,2)   │
├───────────────┼─────────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────

In [12]:
#Query 1：資料規模
# 呼叫 DuckDB 執行 SQL；f-string 讓 {path} 替換成變數值
duckdb.sql(f"""
SELECT COUNT(*) AS n_events,                        -- 總列數
       COUNT(DISTINCT user_id) AS n_users,          -- 不重複 user 數
       COUNT(DISTINCT user_session) AS n_sessions,  -- 不重複 session 數
       MIN(event_time), MAX(event_time)             -- 最早和最晚時間
FROM {path}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬─────────┬────────────┬─────────────────────┬─────────────────────┐
│ n_events │ n_users │ n_sessions │   min(event_time)   │   max(event_time)   │
│  int64   │  int64  │   int64    │      timestamp      │      timestamp      │
├──────────┼─────────┼────────────┼─────────────────────┼─────────────────────┤
│ 42448764 │ 3022290 │    9244421 │ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │
└──────────┴─────────┴────────────┴─────────────────────┴─────────────────────┘



In [7]:
# Query 2：各類 event 的數量和佔比
duckdb.sql(f"""
SELECT event_type, COUNT(*) AS n,           -- event 類型，以及每一類的筆數
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
       -- 每類筆數 ÷ 全部筆數 × 100，四捨五入到小數 2 位
       -- SUM(COUNT(*)) OVER ()：把各組的 COUNT(*) 加總，得到總筆數
FROM {path}
GROUP BY event_type                         -- 依 event_type 分組
ORDER BY n DESC                             -- 依筆數由大到小排序
""").show()

┌────────────┬──────────┬────────┐
│ event_type │    n     │  pct   │
│  varchar   │  int64   │ double │
├────────────┼──────────┼────────┤
│ view       │ 40779399 │  96.07 │
│ cart       │   926516 │   2.18 │
│ purchase   │   742849 │   1.75 │
└────────────┴──────────┴────────┘



In [8]:
# Query 3：Missing 比例
duckdb.sql(f"""
SELECT ROUND(100.0 * AVG(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_category,
       -- CASE WHEN：如果 category_code 是空值就給 1，否則給 0
       -- 對這串 0/1 取平均 = 空值的比例，× 100 變成百分比
       ROUND(100.0 * AVG(CASE WHEN brand IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_brand,
       -- 同樣的邏輯，換成 brand 欄位
       ROUND(100.0 * AVG(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END), 4) AS pct_missing_session
       -- 同樣的邏輯，換成 user_session；取到小數 4 位，因為預期空值極少
FROM {path}
""").show()

┌──────────────────────┬───────────────────┬─────────────────────┐
│ pct_missing_category │ pct_missing_brand │ pct_missing_session │
│        double        │      double       │       double        │
├──────────────────────┼───────────────────┼─────────────────────┤
│                31.84 │              14.4 │                 0.0 │
└──────────────────────┴───────────────────┴─────────────────────┘



找 Missing 比例的目的是決定每個欄位能不能用、要怎麼處理。 Missing 的比例和原因，會直接影響後面的分析和 model。具體有四個用途：

1. 判斷欄位還能不能用
缺很多（例如 category_code 缺 31.8%）就不能當主要分析維度，否則按類別分析時會有將近三分之一的資料落在「NULL」那組，結論會失真。所以我們改用沒有缺值的 category_id。

2. 決定處理方式
刪除有缺值的列、填補（例如填平均數或 'unknown'）、或保留缺值讓 model 自己處理（LightGBM 可以直接處理 NULL）。比例決定選哪個：缺 14.4% 的 brand 如果整列刪掉，會損失 600 多萬筆資料，所以改成填 'unknown'。

3. 檢查缺值是不是隨機的
這是最重要的一點。如果缺值集中在特定情況，就不是隨機的，處理不當會造成偏差。例如：brand 缺值是否集中在便宜的商品或某些類別？如果是，「沒有品牌」本身就帶有資訊，可以做成一個 feature（is_brand_missing）。

4. 寫進 README 的 limitation
Reviewer 會想知道你的資料有哪些缺陷、你怎麼處理。例如「category_code 缺 31.8%，因此類別分析改用 category_id」，這一句就展現了你有檢查資料品質，而不是拿到資料就直接跑 model。

In [ ]:
# Session 層級：有沒有 cart、有沒有 purchase 的交叉表
duckdb.sql(f"""
WITH s AS (
-- 第一段（CTE）：把 event 層級壓成 session 層級，每個 session 一列
    SELECT user_session,
           MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart, -- 每筆 event 是 cart 就給 1，否則 0；
           -- MAX 取組內最大值：只要出現過至少一次 cart 就是 1
           MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM {path}
    GROUP BY user_session  -- 同一個 session 的所有 event 收成一組
)

-- 第二段：對 session 表依 (has_cart, has_purchase) 的四種組合計數
--   0,0 只瀏覽 / 0,1 沒加購物車就購買 / 1,0 加了沒買 / 1,1 加了也買
SELECT has_cart, has_purchase, COUNT(*) AS n_sessions,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM s
GROUP BY has_cart, has_purchase
ORDER BY has_cart, has_purchase
""").show()

# 完全重複的列有幾筆
duckdb.sql(f"""
SELECT (SELECT COUNT(*) FROM {path})
     - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {path})) AS n_duplicate_rows
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────┬────────────┬────────┐
│ has_cart │ has_purchase │ n_sessions │  pct   │
│  int32   │    int32     │   int64    │ double │
├──────────┼──────────────┼────────────┼────────┤
│        0 │            0 │    8333625 │  90.15 │
│        0 │            1 │     337699 │   3.65 │
│        1 │            0 │     281237 │   3.04 │
│        1 │            1 │     291861 │   3.16 │
└──────────┴──────────────┴────────────┴────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┐
│ n_duplicate_rows │
│      int64       │
├──────────────────┤
│            30220 │
└──────────────────┘



In [19]:
# 各 event_type 的重複列數 & 重複比例
duckdb.sql(f"""
WITH d AS (
    SELECT event_type,
           COUNT(*) AS n_rows,
           COUNT(*) - COUNT(DISTINCT (event_time, event_type, product_id, user_id, user_session, price)) AS n_duplicates
    FROM {path}
    GROUP BY event_type
)
SELECT event_type,
       n_rows,
       n_duplicates,
       ROUND(100.0 * n_duplicates / n_rows, 3) AS pct_duplicates   -- 取到小數 3 位，view 的比例很小
FROM d
ORDER BY pct_duplicates DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────┬──────────────┬────────────────┐
│ event_type │  n_rows  │ n_duplicates │ pct_duplicates │
│  varchar   │  int64   │    int64     │     double     │
├────────────┼──────────┼──────────────┼────────────────┤
│ cart       │   926516 │        28073 │           3.03 │
│ purchase   │   742849 │           76 │           0.01 │
│ view       │ 40779399 │         2071 │          0.005 │
└────────────┴──────────┴──────────────┴────────────────┘



重複幾乎全部集中在 cart。View 和 purchase 的重複少到可以忽略，不是整個 tracking 系統都有問題，而是 cart 這個動作特有的現象。最可能的原因是使用者在同一秒內連點「加入購物車」，或是加入多件同樣商品時被記成多筆。兩者在這份資料裡無法區分，但都不是嚴重的資料錯誤。
### 處理決定
不刪除任何資料：重複比例低，且有合理的使用者行為解釋
Label（has_cart、has_purchase）用 MAX 計算，不受重複影響
之後做 cart 相關的計數 feature 時，用 COUNT(DISTINCT product_id)，避免連點造成的灌水
README 的 data quality 段落記一句：cart events 有 3% 為同秒完全重複，判斷為連點或多件加入，未刪除，計數 feature 改用 distinct count